# Lakeflow Connect

## 1.Lakeflow Connect Overview

- LakeFlow Connect streamlines data ingestion with simple, efficient connectors
- It enables you to bring in data from files, cloud storage, databases, enterprise applications, and streaming sources directly into the Databricks Lakehouse
- This makes Databricks source system agnostic

### Key benefit of Lakeflow Connect

- **Managed and Efficient Solution**: Lower costs and quicker time to value
- **Self-Serve Interfaces for Every Practitioner**: Democratized data with an accelerated rate of innovation
- **Unified Observability and Governance**: Secured and healthy pipelines and tables

### LakeFlow Connect Connectors Overview


**Upload Files** <br>
Manual File Uploads allow users to quickly upload local files into volumes or tables.

**Standard Connectors** <br>
Standard Connectors enable ingestion from sources like cloud storage and Kafka using batch, incremental, or streaming modes.

**Managed Connectors**<br>
Managed Connectors are designed for enterprise applications and databases, enabling scalable and efficient incremental data ingestion into the lakehouse.

### Ingestion Methods

#### Batch Ingestion

- Load data as batches of rows into Databricks, often based on a schedule
- Traditional batch ingestion processes all records each time it runs
- Batch ingestion is well-suited for large volumes of historical data where real-time processing is not required
- Common techniques include:
    - The SQL statement: CREATE TABLE AS SELECT
    - The Python method: spark.read.load()

![image_1793853217686.png](./image_1793853217686.png "image_1793853217686.png")

#### Incremental Batch Ingestion

- Only new data is ingested — previously loaded records are skipped automatically
- Provides faster and more resource-efficient ingestion by processing less data
- Common techniques include:
    - The SQL statement: COPY INTO
    - The Python method: spark.readStream (Auto Loader with a timed trigger)
    - Declarative Pipelines: CREATE OR REFRESH STREAMING TABLE

![image_1793853376295.png](./image_1793853376295.png "image_1793853376295.png")

#### Streaming Ingestion

* Continuously load data rows or batches of data rows as they are generated, so you can query them as they arrive in near real-time
* Micro-batch processes small batches at very short, frequent intervals
* Common techniques include:
    * spark.readStream (Auto Loader with continuous trigger)
    * Declarative Pipelines (trigger mode continuous)

![image_1793853502181.png](./image_1793853502181.png "image_1793853502181.png")

#### Types of Lakeflow Connect ingestion

|Category|What it ingests|Examples|
|---|---|---|
|Database connectors (CDC)|Relational database changes|MySQL, PostgreSQL, SQL Server, Oracle|
|SaaS connectors|Enterprise application data|Salesforce, HubSpot, Jira, Workday|
|File connectors|Files and enterprise file content|Google Drive, SharePoint|
|Streaming connectors|Continuous event data|RabbitMQ and supported streaming sources|
|Query-based connectors|Data extracted through source queries|Supported relational databases|
|Direct write|Application-generated records|Zerobus Ingest API|
|Community connectors|Open-source connectors|Community-supported sources|
|Custom connectors|Your own source integration|Proprietary APIs and databases|

### Delta Lake Overview

- Delta Lake delivers open, reliable, and scalable data management for the Lakehouse.
- It offers ACID transactions, time travel, schema enforcement, and support for both batch and streaming workloads.

**Ingesting Data Into Delta Lake**
* Delta Lake is an open-source protocol for reading and writing files to cloud storage. 
* UC tables offer an open table format that supports the Lakehouse architecture and storage on AWS, Azure, and GCP cloud data lakes.

**UC Table (Conceptual View)**
- Data is stored in UC tables.
- Internally, these tables store data as files within a folder directory.

![image_1793859045035.png](./image_1793859045035.png "image_1793859045035.png")

**UC Table (Technical View)**
- Data is stored as Parquet files in the directory.
- Delta Lake adds JSON-based Delta logs alongside these files.
- Delta logs track transactions and table versions.

![image_1793859101968.png](./image_1793859101968.png "image_1793859101968.png")

**UC table Key Features**
- _ACID Transactions_: Atomicity, Consistency, Isolation, Durability for all operations, allowing multiple users to read and write data concurrently without conflicts.
- _Data Manipulation Language (DML)_: Supports DML operations such as INSERT, UPDATE, DELETE, and MERGE, enabling flexible data management.
- _Time Travel_: Allows users to query and revert to previous versions of data, facilitating auditing and recovery.
- _Schema Evolution and Enforcement_: Enforces a defined schema for data integrity while allowing schema evolution, **enabling structural changes without breaking existing workflows**.
- Unified batch and streaming processing, optimization and performance, and scalability — and Delta Lake is open source.

## 2.Data Ingestion Patters From Cloud Object Storage

![image_1793883136710.png](./image_1793883136710.png "image_1793883136710.png")

### CREATE TABLE AS (CTAS)

* **Supported file formats**: Supports reading file formats like:
JSON | CSV | XML | TEXT | BINARYFILE | PARQUET | AVRO | ORC
* **Automatic format detection**: Can automatically detect the file format in supported scenarios, so specifying the format option is not always necessary.
* **Schema inference**: Can infer a unified schema across multiple files within the specified location.
* **Explicit schema**: Supports defining the schema manually using the schema option.
* **Format-specific options**: Supports specifying file-format-specific options such as CSV headers, delimiters, JSON parsing options and malformed-record handling.
* **Malformed record handling**: Supports rescued data columns and applicable parsing options to handle malformed or unexpected records.
* **Schema evolution**: In streaming mode, supports schema evolution when new columns are discovered, according to the configured schema evolution mode.



#### CREATE TABLE AS
* CREATE TABLE AS (CTAS) creates a Delta table by default from files in cloud object storage. 


####  read_files
* read_files() function is an important Databricks SQL table-valued function (TVF) used to reads files under a provided location and returns the data in tabular form.
* It is particularly useful in data engineering because it allows you to ingest files into your lakehouse without having to create a separate file-reading pipeline for every format or source directory.
* read_files() can read files in batch queries and streaming queries.


#### Syntax:
```
CREATE TABLE new_table AS
          SELECT *
          FROM read_files(
            <path_to_file(s)>,
            format => '<file_type>',
            <other_format_specific_options>
          );    
```

#### Example

In [0]:
%sql
USE databrickstutorial.dlh; 
CREATE OR REPLACE TABLE employees AS
SELECT *,
_metadata.file_path AS file_path
FROM 
read_files(
    '/Volumes/databrickstutorial/dlh/dbo/CSV/Employees.csv',
    format => 'csv'
)

In [0]:
%sql
SELECT * FROM databrickstutorial.dlh.employees LIMIT 5

|EMPLOYEEID|NAME_FIRST|NAME_MIDDLE|NAME_LAST|NAME_INITIALS|SEX|LANGUAGE|PHONENUMBER|EMAILADDRESS|LOGINNAME|ADDRESSID|VALIDITY_STARTDATE|VALIDITY_ENDDATE|_rescued_data|file_path|
|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|
|1|Derrick|L|Magill|null|M|E|630-374-0306|derrick.magill@itelo.info|derrickm|1000000001|20000101|99991231|null|dbfs:/Volumes/databrickstutorial/dlh/dbo/CSV/Employees.csv|
|2|Philipp|T|Egger|null|M|E|09603 61 24 64|philipp.egger@itelo.info|philippm|1000000002|20000101|99991231|null|dbfs:/Volumes/databrickstutorial/dlh/dbo/CSV/Employees.csv|
|3|Ellis|K|Robertson|null|M|E|070 8691 2288|ellis.robertson@itelo.info|ellism|1000000003|20000101|99991231|null|dbfs:/Volumes/databrickstutorial/dlh/dbo/CSV/Employees.csv|
|4|William|M|Mussen|null|M|E|026734 4556|william.mussen@itelo.info|williamm|1000000004|20000101|99991231|null|dbfs:/Volumes/databrickstutorial/dlh/dbo/CSV/Employees.csv|
|5|Javas|null|Hegde|null|M|E|2224135120|javas.hegde@itelo.info|javasm|1000000005|20000101|99991231|null|dbfs:/Volumes/databrickstutorial/dlh/dbo/CSV/Employees.csv|

### COPY INTO

- Copy files from cloud storage into the Delta table.
- Performs a bulk load from files in cloud into the table
- FROM clause specifies the location of the CSV files
- COPY INTO is ideal for situations where the cloud storage location is continuously adding files
- Key aspects of COPY INTO:
    - Idempotent: Will skip any files that have already been loaded into the table; only new files will be ingested
    - File format support: Parquet, JSON, XML, and others
    - FROM clause: Specifies the path of the cloud storage location where new files are being continuously added
    - FORMAT_OPTIONS(): Controls how the source files are parsed and interpreted (options depend on file format)
    - COPY_OPTIONS(): Controls the behavior of the COPY INTO operation itself, such as:
        - Schema evolution using (mergeSchema)
        - Idempotency using (force)


##### Syntax:
```
CREATE TABLE new_table <metadata_definition>;
        
COPY INTO new_table
FROM '<dir_path>'
FILEFORMAT = <file_type>
FORMAT_OPTIONS (<options>)
COPY_OPTIONS (<options>)
```

In [0]:
%sql
USE databrickstutorial.dlh; 

CREATE OR REPLACE TABLE business_partner (
    PARTNERID INT,
    PARTNERROLE INT,
    EMAILADDRESS STRING,
    PHONENUMBER BIGINT,
    FAXNUMBER STRING,
    WEBADDRESS STRING,
    ADDRESSID INT,
    COMPANYNAME STRING,
    LEGALFORM STRING,
    CREATEDBY INT,
    CREATEDAT INT,
    CHANGEDBY INT,
    CHANGEDAT INT,
    CURRENCY STRING
)
USING DELTA; 

USE databrickstutorial.dlh; 

COPY INTO business_partner
FROM '/Volumes/databrickstutorial/dlh/dbo/CSV/BusinessPartners.csv'
FILEFORMAT = CSV
FORMAT_OPTIONS (
  'header' = 'true',
  'inferSchema' = 'true',
  'nullValue' = ''        -- Treats empty CSV fields as true database NULLs instead of strings
);


In [0]:
%sql
SELECT * FROM business_partner LIMIT 5;

|PARTNERID|PARTNERROLE|EMAILADDRESS|PHONENUMBER|FAXNUMBER|WEBADDRESS|ADDRESSID|COMPANYNAME|LEGALFORM|CREATEDBY|CREATEDAT|CHANGEDBY|CHANGEDAT|CURRENCY|
|---|---|---|---|---|---|---|---|---|---|---|---|---|---|
|100000000|2|maria.brown@all4bikes.com|622734567|null|http://www.all4bikes.com|1000000034|All For Bikes|Inc.|10|20181003|10|20181003|USD|
|100000001|2|bob.buyer@amazebikes.com|3088530|null|http://www.amazebikes.com|1000000035|Amaze Bikes Inc|Inc.|13|20181003|13|20181003|USD|
|100000002|2|victor.sanchez@arenasportusa.com|3023352668|null|http://www.arenasportusa.com|1000000036|Arena Sports Inc|Inc.|14|20181003|14|20181003|USD|
|100000003|2|franklin.jones@atlantausa.com|511403266|null|http://www.atlantausa.com|1000000037|Atlanta Corp Inc|Inc.|10|20181003|10|20181003|USD|
|100000004|2|robert_brown@bikeworlds.com|2244668800|null|http://www.bikeworlds.com|1000000038|Bike World Inc|Inc.|4|20181003|4|20181003|USD|

### AUTO LOADER

* Incremental batch or streaming ingestion using Auto Loader
* Process new data files incrementally as they arrive
* Automatically detect and load new files into Delta tables
* Simplify handling of incremental and streaming data
* Use with both Python and SQL (via Declarative Pipelines)
* Scale to process billions of files
* Rely on Spark Structured Streaming for efficient and reliable ingestion

#### Syntax

**Autoloader with PySpark**
* We start with .readSteam and set the format to "`cloudFiles`", which enables AutoLoader
* Then, we specify the file format as `JSON`, and define the `schema location` using `cloudFiles.schemaLocation`. 
* Next, we use .load() to point to the `location of the files`, in this case a path under `/Volumes` referencing Unity Catalog.
* On the write side, we configure `.writeStream with a checkpoint location to maintain state and progress`, and set a trigger interval of every 5 seconds.
* Finally, we use `.toTable()` to `write the data into a Delta table` specified by catalog, database, and table name.

Syntax:
```
 (
        spark.readStream

            # Use Databricks Auto Loader instead of the standard file streaming source.
            .format("cloudFiles")

            # Specify the format of the incoming files. Examples: "json", "csv", "parquet", "avro"
            .option("cloudFiles.format", source_format)

            # Provide the schema that should be used to interpret the incoming data.
            .schema(<schema>)

            # Location where Auto Loader stores schema information and tracks schema evolution.
            .option("cloudFiles.schemaLocation", <checkpoint_directory>)

            # Location of the input files.
            .load(<data_source>)

            # Convert the streaming DataFrame into a streaming write operation.
            .writeStream

            # Checkpoint location used by Structured Streaming to maintain progress/state and provide fault tolerance.
            .option("checkpointLocation", <checkpoint_directory>)

            # Allow the target schema to evolve when new columns are detected.
            .option("mergeSchema", "true")

            # Process all currently available files as a batch, then stop the streaming query.
            .trigger(availableNow=True)

            # Write the incoming data to the specified Delta table.
            .toTable(<table_name>)
    )
```


**Auto Loader with Databricks SQL**
* Databricks recommends using streaming tables to ingest data with Databricks SQL (instead of COPY INTO). 
* A streaming table is a table registered to Unity Catalog that includes `additional support for streaming or incremental` data processing.
    * When you create a streaming table, a `pipeline is automatically generated` for it.
    * Steaming tables can be used for incremental data loading from both `Kafka and cloud object storage`.
* To create a streaming table from files in a volume, you use Auto Loader. Databricks recommends using `Auto Loader with Apache Spark Declarative Pipelines` for most data ingestion tasks from cloud object storage. Together, Auto Loader and Declarative Pipelines are designed to incrementally and idempotently load continuously growing datasets as they arrive.
* Streaming tables in Databricks SQL are backed by `serverless Spark Declarative Pipelines`.
* To use Auto Loader in Databricks SQL, use the read_files function with the `STREAM` keyword in the FROM clause.


Syntax:
```
CREATE OR REFRESH STREAMING TABLE
  catalog.schema.table
SCHEDULE EVERY 1 HOUR
AS
SELECT *
FROM STREAM read_files(
  '<dir_path>',
  format => '<file_type>'
)
```

#### Example

In [0]:
( 
    spark.
    readStream.
    format('cloudFiles').
    option('cloudFiles.format', 'csv').
    option("cloudFiles.schemaLocation", "/Volumes/databrickstutorial/dlh/dbo/CheckPoint/").
    load('/Volumes/databrickstutorial/dlh/dbo/ProductsFiles/').
    writeStream.
    option("checkpointLocation", "/Volumes/databrickstutorial/dlh/dbo/CheckPoint/").
    trigger(availableNow=True).
    toTable('Products')
)


In [0]:
%sql
SELECT * FROM products LIMIT 5;

|PRODUCTID|TYPECODE|PRODCATEGORYID|CREATEDBY|CREATEDAT|CHANGEDBY|CHANGEDAT|SUPPLIER_PARTNERID|TAXTARIFFCODE|QUANTITYUNIT|WEIGHTMEASURE|WEIGHTUNIT|CURRENCY|PRICE|WIDTH|DEPTH|HEIGHT|DIMENSIONUNIT|PRODUCTPICURL|_rescued_data|
|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|
|DB-1081|PR|DB|10|20181003|10|20181003|100000023|1|EA|13.6|KG|USD|1499|null|null|null|null|null|null|
|DB-1082|PR|DB|4|20181003|4|20181003|100000024|1|EA|14.3|KG|USD|1250|null|null|null|null|null|null|
|DB-1083|PR|DB|6|20181003|6|20181003|100000025|1|EA|15.4|KG|USD|1199|null|null|null|null|null|null|
|CC-1021|PR|CC|10|20181003|10|20181003|100000009|1|EA|8.1|KG|USD|1144|null|null|null|null|null|null|
|CC-1022|PR|CC|13|20181003|13|20181003|100000010|1|EA|8.0|KG|USD|1200|null|null|null|null|null|null|

In [0]:
%sql
USE databrickstutorial.dlh;

CREATE OR REFRESH STREAMING TABLE
  databrickstutorial.dlh.SalesOrders
SCHEDULE EVERY 1 HOUR
AS
SELECT *
FROM STREAM read_files(
  '/Volumes/databrickstutorial/dlh/dbo/CSV/SalesOrders/',
  format => 'csv'
)

In [0]:
%sql
SELECT * FROM SalesOrders LIMIT 5;

|SALESORDERID|CREATEDBY|CREATEDAT|CHANGEDBY|CHANGEDAT|FISCVARIANT|FISCALYEARPERIOD|NOTEID|PARTNERID|SALESORG|CURRENCY|GROSSAMOUNT|NETAMOUNT|TAXAMOUNT|LIFECYCLESTATUS|BILLINGSTATUS|DELIVERYSTATUS|_rescued_data|
|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|
|500000000|4|20180111|4|20180116|K4|2018001|null|100000022|APJ|USD|13587|11888.625|1698.375|C|C|C|null|
|500000001|2|20180112|2|20180115|K4|2018001|null|100000026|EMEA|USD|12622|11044.25|1577.75|C|C|C|null|
|500000002|5|20180115|5|20180120|K4|2018001|null|100000018|APJ|USD|45655|39948.125|5706.875|C|C|C|null|
|500000003|3|20180115|3|20180120|K4|2018001|null|100000009|EMEA|USD|101786|89062.75|12723.25|C|C|C|null|
|500000004|8|20180116|8|20180117|K4|2018001|null|100000025|EMEA|USD|71684|62723.5|8960.5|C|C|C|null|